# Ejercicio 5 - scikit-learn Pipeline con MLflow

Notebook autosuficiente para Jupyter y Databricks. Registra un pipeline mediante **MLflow Experiments** y **MLflow Models**.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# En Databricks, MLflow usa automaticamente el tracking del workspace.
mlflow.set_experiment('/Shared/ejercicio-5-sklearn-mlflow')

wine = load_wine(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.25, random_state=42, stratify=wine.target
)
params = {'n_estimators': 150, 'max_depth': 5, 'random_state': 42}
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(**params)),
])


In [ ]:
with mlflow.start_run(run_name='random-forest-wine-notebook'):
    mlflow.log_params({f'rf_{name}': value for name, value in params.items()})
    mlflow.log_param('test_size', 0.25)
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    metrics = {
        'accuracy': accuracy_score(y_test, predictions),
        'f1_weighted': f1_score(y_test, predictions, average='weighted'),
    }
    mlflow.log_metrics(metrics)
    model_info = mlflow.sklearn.log_model(
        sk_model=pipeline,
        name='wine_random_forest_notebook',
        input_example=X_train.head(3),
        serialization_format='cloudpickle',
    )
    mlflow.set_tags({'exercise': '5', 'source': 'databricks-notebook'})

print({name: round(value, 4) for name, value in metrics.items()})
print('Modelo registrado:', model_info.model_uri)


## Entrega

Una vez importado y ejecutado en Databricks, copie la URL del navegador. Esa es la URL que debe incluirse en el PDF como enlace al notebook del workspace.